In [ ]:
# Dataset description ##
# M plants  = 250 total systems. 200 train, 50 test
# M x 4 component systems installed in M plants
# Each COMPONENT has 10 sensors 
# All sensors read at 1 atu for all components  stored as  [10 x 1000]
# --------------------------------------------------------------------------------------------------
# Plant 1..M
# sensor   s1      s2    s3       s4     s5     s6     s7      s8      s9     s10      Label
# COMP1  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# COMP2  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# COMP3  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1 
# COMP4  [1000] [1000] [1000]  [1000]  [1000] [1000] [1000]  [1000]  [1000]  [1000]     0/1
# --------------------------------------------------------------------------------------------------
# Challenge: component health-state detection using multivariate sensor time-series.
# each element in the train data array is a MATLAB struct [('trajectory', 'O'), ('Label', 'O')]
# each timestamp tau has a label


#### Verify environment

In [ ]:
try:
    import google.colab
    print("Running on Colab")
    from google.colab import drive
    drive.mount('/content/drive')

except:
    print("Running locally")
!ls /content/drive/MyDrive/hh_project

#### Load data

In [ ]:
import scipy.io.matlab as sciom
import os
print(os.getcwd())
data = sciom.loadmat("/content/drive/MyDrive/hh_project/data/train.mat")
#data = sciom.loadmat("/home/vinayp/dev/hh_project/data/train.mat")
print(data.keys()) 


#### Check data structure

In [ ]:
# Whole train data
train_d = data['Train_data']
print(f"ad shape:{train_d.shape},ad dtype:{train_d.dtype}") 

# Get sensor values of all plants
iplant_d = train_d['trajectory']
print(f"apd shape:{iplant_d.shape}, apd dtype:{iplant_d.dtype}") 

#Get labels for all plants
label_d = train_d['Label']
print(f"ald shape:{label_d.shape}, ald dtype:{label_d.dtype}") 

# Get single plant data
plant1 = iplant_d[0][0]
print(f"spd shape:{plant1.shape}, spd dtype:{plant1.dtype}") 

# Get single plant labels
label1 = label_d[0][0]  #(1,4)
print(f"sld shape:{plant1.shape}, sld dtype:{plant1.dtype}") 

# Get single component data
comp1 = plant1[0][0]
print(f"scd shape:{comp1.shape}")
# Get single component label
clabel1 = label1[0]
print(f"cld shape:{clabel1.shape}")
#print(f"cld shape:{clabel1.shape[1]}")

import numpy as np

# Check abnormality time
for t in range(label1.shape[0]):
    y = label1[t]
    idx = np.where(y == 1)[0]
    if len(idx) > 0:
        print(f"component {t} abnormal at t = {idx[0]}")
    else:
        print(f"component {t} was never abnormal")

for sens in range(comp1.shape[0]):
    y = comp1[sens]
    maxt = np.max(y)
    min = np.min(y)
    print(f"max Sens{sens}:{maxt}, min Sens{sens}:{min}")





#### Graphical view of component

In [ ]:
import plotly.graph_objects as pgo
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = "notebook"
sensorx = comp1[0]
labelc1 = clabel1
fig = make_subplots(rows=11, cols=1, shared_xaxes=True, vertical_spacing=0.02, subplot_titles=("Sensor Reading", "label"))

for sns in range(0, comp1.shape[0]):
    sensorx = comp1[sns]

    fig.add_trace(
        pgo.Scatter(
            y = sensorx,
            mode = "lines",
            name = "sensor",
            hovertemplate='Time: %{x}<br>Value:%{y}<extra></extra>'
        ),
        row=sns+1,
        col=1
    )

fig.add_trace(
    pgo.Scatter(
        y = labelc1,
        mode = 'lines',
        name = 'Label',
        hovertemplate = 'Time: %{x}<br>Label:%{y}<extra></extra>'
    ),
    row = 11,
    col = 1
)

fig.update_layout(
    height = 2000,
    title = "Sensor and Label visualization",
    hovermode="x unified"
)

# Axis labels
fig.update_xaxes(title_text="Time (atu)", row=2, col=1)
fig.update_yaxes(title_text="sensor_reading", row=1, col=1)
fig.update_yaxes(title_text="label", row=2, col=1)

fig.show()

##### Experiment: Train a model to predict health of component or plant and identify malfunction